# 知识图谱可视化

读取 `memory.jsonl`，用 **yFiles Graphs for Jupyter** 以图数据库节点-边的方式展示记忆内容。

所有图表直接在 Notebook cell 中交互式渲染。

## 1. 导入依赖 & 加载数据

In [ ]:
import json
import textwrap
import networkx as nx
from yfiles_jupyter_graphs import GraphWidget

print('✅ 依赖加载成功')

In [ ]:
MEMORY_FILE = '../memory.jsonl'

entities = {}
relations = []

with open(MEMORY_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        if record.get('type') == 'entity':
            entities[record['name']] = {
                'entityType': record.get('entityType', 'unknown'),
                'observations': record.get('observations', [])
            }
        elif record.get('type') == 'relation':
            relations.append({
                'from': record['from'],
                'to': record['to'],
                'relationType': record.get('relationType', '')
            })

print(f'实体数: {len(entities)}')
print(f'关系数: {len(relations)}')
for name, info in entities.items():
    print(f'  [{info["entityType"]}] {name}  ({len(info["observations"])} 条观察)')

## 2. 构建 NetworkX 有向图

In [ ]:
TYPE_COLORS = {
    'project':         '#4A90D9',
    'design-decision': '#E8A838',
    'technique':       '#50C878',
    'analysis':        '#E06C75',
    'research':        '#C678DD',
    'configuration':   '#56B6C2',
    'design-pattern':  '#D19A66',
    'tool-option':     '#98C379',
}
DEFAULT_COLOR = '#ABB2BF'

def get_color(t):
    return TYPE_COLORS.get(t, DEFAULT_COLOR)

G = nx.DiGraph()
for name, info in entities.items():
    G.add_node(name,
               entityType=info['entityType'],
               obs_count=len(info['observations']),
               observations='\n'.join(f'{i}. {o}' for i, o in enumerate(info['observations'], 1)))

for rel in relations:
    if rel['from'] in entities and rel['to'] in entities:
        G.add_edge(rel['from'], rel['to'], label=rel['relationType'])

print(f'图节点数: {G.number_of_nodes()}, 图边数: {G.number_of_edges()}')

## 3. 🌟 yFiles 交互式全局图谱

直接在 cell 中渲染，支持拖拽、缩放、点击节点查看详情。

In [ ]:
w = GraphWidget(graph=G)

# 使用 hierarchic 布局 (层次结构)
w.graph_layout = 'hierarchic'

# 节点颜色: 按 entityType 映射
def custom_node_color(node):
    et = node['properties'].get('entityType', 'unknown')
    return get_color(et)

w.node_color_mapping = custom_node_color

# 节点大小: 按观察数缩放
def custom_node_scale(node):
    oc = node['properties'].get('obs_count', 1)
    return 1.0 + oc * 0.15

w.node_scale_factor_mapping = custom_node_scale

# 节点标签: 显示名称
def custom_node_label(node):
    return node['properties'].get('label', str(node['id']))

w.node_label_mapping = custom_node_label

# 边标签: 显示关系类型
def custom_edge_label(edge):
    return edge['properties'].get('label', '')

w.edge_label_mapping = custom_edge_label

# 开启邻居高亮
w.context_start_with = 'Neighborhood'

# 显示
display(w)

## 4. 子图聚焦

修改 `CENTER_NODE` 查看不同节点的局部关系。

In [ ]:
CENTER_NODE = 'Splunk-MCP-Design'  # ← 修改这里
DEPTH = 1

if CENTER_NODE not in G:
    print(f'节点 "{CENTER_NODE}" 不存在。可用节点:')
    for n in sorted(G.nodes): print(f'  - {n}')
else:
    neighbors = set()
    layer = {CENTER_NODE}
    for _ in range(DEPTH):
        nxt = set()
        for nd in layer:
            nxt.update(G.successors(nd))
            nxt.update(G.predecessors(nd))
        neighbors.update(nxt)
        layer = nxt
    neighbors.add(CENTER_NODE)
    sub = G.subgraph(neighbors).copy()

    w2 = GraphWidget(graph=sub)
    w2.graph_layout = 'hierarchic'
    w2.node_color_mapping = custom_node_color
    w2.node_scale_factor_mapping = custom_node_scale
    w2.node_label_mapping = custom_node_label
    w2.edge_label_mapping = custom_edge_label

    print(f'中心: {CENTER_NODE}, 深度: {DEPTH}, 节点: {sub.number_of_nodes()}, 边: {sub.number_of_edges()}')
    display(w2)

## 5. 层次布局视图

用层次布局展示图谱的方向性关系。

In [ ]:
w3 = GraphWidget(graph=G)
w3.graph_layout = 'hierarchic'
w3.node_color_mapping = custom_node_color
w3.node_scale_factor_mapping = custom_node_scale
w3.node_label_mapping = custom_node_label
w3.edge_label_mapping = custom_edge_label
display(w3)

## 6. 节点详情查看器

In [ ]:
def inspect_node(name):
    if name not in entities:
        print(f'实体 "{name}" 不存在。可用实体:')
        for n in sorted(entities.keys()): print(f'  - {n}')
        return
    info = entities[name]
    print(f'═══ {name} ═══')
    print(f'类型: {info["entityType"]}')
    print(f'\n观察 ({len(info["observations"])} 条):')
    for i, obs in enumerate(info['observations'], 1):
        print(f'  {i}. {obs}')
    out_rels = [r for r in relations if r['from'] == name]
    if out_rels:
        print(f'\n出向关系 ({len(out_rels)}):')
        for r in out_rels: print(f'  → [{r["relationType"]}] → {r["to"]}')
    in_rels = [r for r in relations if r['to'] == name]
    if in_rels:
        print(f'\n入向关系 ({len(in_rels)}):')
        for r in in_rels: print(f'  {r["from"]} → [{r["relationType"]}] →')

inspect_node('IDE-Data-Agent')

## 7. 图统计

In [ ]:
from collections import Counter

print('=== 图统计 ===')
print(f'节点数: {G.number_of_nodes()}')
print(f'边数:   {G.number_of_edges()}')
print(f'密度:   {nx.density(G):.4f}')
print()

type_counts = Counter(G.nodes[n].get('entityType', 'unknown') for n in G.nodes)
print('节点类型分布:')
for t, c in type_counts.most_common(): print(f'  {t}: {c}')
print()

print('节点度数排名:')
for name, deg in sorted(G.degree(), key=lambda x: x[1], reverse=True):
    print(f'  {name}: 总度={deg} (入={G.in_degree(name)}, 出={G.out_degree(name)})')
print()

print('观察条数排名:')
for name, info in sorted(entities.items(), key=lambda x: len(x[1]['observations']), reverse=True):
    print(f'  {name}: {len(info["observations"])} 条')